# crystal.py

Collection of tools to define and work with crsyta structures
1- dictionary: species
2- class crystal
3- funtion 

## Dictionary species

This is a dictionary of dictionaries. Each key is the name of a crystal structure. The associated value is a dictionary of the following properties

- `'type'`: Geometry type ('linear', 'hexagonal', 'FFC', etc)
- `'direct vectors'`: list of ndarrays of lattice dimensions or None (if reciprocal lattice vectors are given below)
- `'direct lattice unit'`: float
- `'reciprocal vectors'`: list of ndarrays of lattice dimensions or None (if direct lattice vectors are given above)
-`'reciprocal symmetry points'`: dictionary of name of symmetry point ('Gamma','K', etc) as keys and a ndarray of the coordinates in the reciprocal basis
-`'orbitals'`: lsit of dictionaries. Each dictionary refers to an orbital in the base. The keys are
    - `'name'`: orbital name
    - `'position'`: orbital position in the direct vector basis
    - `'element'`: chemical element
    - `'orbital'`: orbital type ('pz', etc)
    - `'energy'`: float, energy of the orbital
    - `'neighbors'`: list of dictionaries each describing close orbital neighbor. Each neighbor dictionary has the following keys
        - `'name'`: orbital name
        - `'delta'`: ndarray of crystal dimension, spatial shift from of the neigbor orbital, relative to the position of the current orbital
        - `'hopping'`: hopping energy

Example: Graphene

```python
import numpy as np
import crystal as cr
from pprint import pprint

pprint(cr.species['graphene'])

# {'direct lattice unit': 2.46e-10,
#  'direct vectors': [array([0.8660254, 0.5      ]),
#                     array([ 0.8660254, -0.5      ])],
#  'orbitals': [{'element': 'C',
#                'energy': -0.28,
#                'name': 'A',
#                'neighbors': [{'delta': array([-0.33333333, -0.33333333]),
#                               'hopping': -2.97,
#                               'name': 'B'},
#                              {'delta': array([ 0.66666667, -0.33333333]),
#                               'hopping': -2.97,
#                               'name': 'B'},
#                              {'delta': array([-0.33333333,  0.66666667]),
#                               'hopping': -2.97,
#                               'name': 'B'}],
#                'orbital': 'pz',
#                'position': array([-0.16666667, -0.16666667])},
#               {'element': 'C',
#                'energy': -0.28,
#                'name': 'B',
#                'neighbors': [{'delta': array([0.33333333, 0.33333333]),
#                               'hopping': -2.97,
#                               'name': 'A'},
#                              {'delta': array([-0.66666667,  0.33333333]),
#                               'hopping': -2.97,
#                               'name': 'A'},
#                              {'delta': array([ 0.33333333, -0.66666667]),
#                               'hopping': -2.97,
#                               'name': 'A'}],
#                'orbital': 'pz',
#                'position': array([0.16666667, 0.16666667])}],
#  'reciprocal symmetry points': {'Gamma': array([0, 0]),
#                                 'Gamma01': array([1, 1]),
#                                 'Gamma10': array([1, 0]),
#                                 'K': array([0.66666667, 0.33333333]),
#                                 'Kp': array([0.33333333, 0.66666667]),
#                                 'Kpv': array([ 0.33333333, -0.33333333]),
#                                 'M': array([0.5, 0.5])},
#  'reciprocal vectors': [array([0.57735027, 1.        ]),
#                         array([ 0.57735027, -1.        ])],
#  'type': 'hexagonal'}

## Class Crystal

Class defining a crystal object. It has the crystal parameters as well as the grid definition

### Attributes

- `crystal_species`: str. Name od the crystal species
- `crystal_parameters`: dictionary of species entry
- `direct_lattice_unit`
- `'direct_vector_coordinates'`: list of ndarrays arch with the coordinates of an element of the direct vector space.
- `'reciprocal_vector_coordinates'`: list of ndarrays arch with the coordinates of an element of the direct vector space.
- `grid`: Object of class Grid
    - `'ndim'`: int, dimensions of the grid
    - `'unit'`: float spatial unit
    - `'origin'`: ndarray of diemnsion ndim with the coordinate origin
    - `'limits'`: list of ndim ndarrays with the relative extension of the grid from the origin
    - `'nptx'`: list of ndim int, number of points in each axis
    - `'filter'`: callable, filter of the active points in the grid
    - `'points`': 1D list of the active points in the grid


Let us make an example. For the case of Graphene we have 

<div style="text-align:center;">
  <img src="figures/graphene lattices.png" alt="Diagram" style="width:1000px;">
  <figcaption>Figure 1: Graphene lattice structure (left: direct space, right: reciprocal).</figcaption>
</div>

We can explore the attributes

```python
import numpy as np
from grid import UniformCartesianGrid
from crystal import crystal, species, vector_in_cart
from pprint import pprint

cr= crystal(species['graphene'], space='reciprocal')

print(f"{cr.direct_lattice_unit=}")
print(f"{cr.direct_vectors=} in units of $a$ (direct lattice unit) NOTE: $\\sqrt 3 /2 = 0.8660254$")
print(f"{cr.reciprocal_vectors=} in units of $2 \\pi/a$")
print()
pprint(cr.parameters)
print()

Kp=cr.parameters['reciprocal symmetry points']['Kp']

print(f'{vector_in_cart(Kp, basis=cr.reciprocal_vectors)=}')

cr.direct_lattice_unit=2.46e-10
cr.direct_vectors=[array([0.8660254, 0.5      ]), array([ 0.8660254, -0.5      ])] in units of $a$ (direct lattice unit) NOTE: $\sqrt 3 /2 = 0.8660254$
cr.reciprocal_vectors=[array([0.57735027, 1.        ]), array([ 0.57735027, -1.        ])] in units of $2 \pi/a$

# {'direct lattice unit': 2.46e-10,
#  'direct vectors': [array([0.8660254, 0.5      ]),
#                     array([ 0.8660254, -0.5      ])],
#  'orbitals': [{'element': 'C',
#                'energy': -0.28,
#                'name': 'A',
#                'neighbors': [{'delta': array([-0.33333333, -0.33333333]),
#                               'hopping': -2.97,
#                               'name': 'B'},
#                              {'delta': array([ 0.66666667, -0.33333333]),
#                               'hopping': -2.97,
#                               'name': 'B'},
#                              {'delta': array([-0.33333333,  0.66666667]),
#                               'hopping': -2.97,
#                               'name': 'B'}],
#                'orbital': 'pz',
#                'position': array([-0.16666667, -0.16666667])},
#               {'element': 'C',
#                'energy': -0.28,
#                'name': 'B',
#                'neighbors': [{'delta': array([0.33333333, 0.33333333]),
#                               'hopping': -2.97,
#                               'name': 'A'},
#                              {'delta': array([-0.66666667,  0.33333333]),
#                               'hopping': -2.97,
#                               'name': 'A'},
#                              {'delta': array([ 0.33333333, -0.66666667]),
#                               'hopping': -2.97,
#                               'name': 'A'}],
#                'orbital': 'pz',
#                'position': array([0.16666667, 0.16666667])}],
#  'reciprocal symmetry points': {'Gamma': array([0, 0]),
#                                 'Gamma01': array([1, 1]),
#                                 'Gamma10': array([1, 0]),
#                                 'K': array([0.66666667, 0.33333333]),
#                                 'Kp': array([0.33333333, 0.66666667]),
#                                 'Kpv': array([ 0.33333333, -0.33333333]),
#                                 'M': array([0.5, 0.5])},
#  'reciprocal vectors': [array([0.57735027, 1.        ]),
#                         array([ 0.57735027, -1.        ])],
#  'type': 'hexagonal'}

# vector_in_cart(Kp, basis=cr.reciprocal_vectors)=array([ 0.85911676, -0.47421658])
```

